# CRM Pipeline: Pull → Dedup → Enrich → Report

## What this shows

Build a deterministic CRM pipeline from fixture records: normalize Salesforce/HubSpot-shaped contacts into canonical CRM models, deduplicate across systems, prepare address rows for geocoding, and reshape opportunities into report-ready tables.

This is a pure/offline notebook. It does not construct Salesforce or HubSpot connectors, read credentials, call APIs, or write back to any CRM.


## 1. Canonical fixture records

The live connectors expose vendor-specific records. Governed execution starts after extraction, with tiny fixture records mapped into the vendor-neutral `CRMContact`, `CRMAccount`, and `CRMOpportunity` models.


In [ ]:
from datetime import date
from decimal import Decimal

import pandas as pd

from siege_utilities.connectors import CRMAccount, CRMAddress, CRMContact, CRMOpportunity

sf_contacts = [
    CRMContact(
        id="sf-c-001",
        first_name="Ada",
        last_name="Lovelace",
        email="ada@example.test",
        account_id="sf-a-001",
        address=CRMAddress(street="100 Congress Ave", city="Austin", state="TX", postal_code="78701", country="US"),
        source_system="salesforce",
        source_id="003SF001",
        metadata={"lead_score": 92},
    ),
    CRMContact(
        id="sf-c-002",
        first_name="Grace",
        last_name="Hopper",
        email="grace@example.test",
        account_id="sf-a-002",
        address=CRMAddress(street="200 Main St", city="Houston", state="TX", postal_code="77002", country="US"),
        source_system="salesforce",
        source_id="003SF002",
        metadata={"lead_score": 88},
    ),
]

hs_contacts = [
    CRMContact(
        id="hs-c-100",
        first_name="Ada",
        last_name="Lovelace",
        email="ada@hubspot.example.test",
        account_id="hs-a-100",
        address=CRMAddress(street="100 Congress Avenue", city="Austin", state="TX", postal_code="78701", country="US"),
        source_system="hubspot",
        source_id="101",
        metadata={"lifecycle_stage": "opportunity"},
    ),
    CRMContact(
        id="hs-c-101",
        first_name="Katherine",
        last_name="Johnson",
        email="katherine@example.test",
        account_id="hs-a-101",
        address=CRMAddress(street="1 NASA Pkwy", city="Houston", state="TX", postal_code="77058", country="US"),
        source_system="hubspot",
        source_id="102",
        metadata={"lifecycle_stage": "customer"},
    ),
]

accounts = [
    CRMAccount(id="sf-a-001", name="Analytical Engines LLC", revenue=Decimal("2500000"), employee_count=42, source_system="salesforce", source_id="001SF001"),
    CRMAccount(id="hs-a-101", name="Orbital Math Lab", revenue=Decimal("1750000"), employee_count=25, source_system="hubspot", source_id="201"),
]

opportunities = [
    CRMOpportunity(id="sf-o-001", name="Austin field program", stage="Proposal", amount=Decimal("125000"), close_date=date(2026, 3, 15), probability=0.45, source_system="salesforce", source_id="006SF001"),
    CRMOpportunity(id="sf-o-002", name="Houston GOTV analytics", stage="Negotiation", amount=Decimal("220000"), close_date=date(2026, 4, 1), probability=0.7, source_system="salesforce", source_id="006SF002"),
    CRMOpportunity(id="hs-o-100", name="Volunteer data mart", stage="Proposal", amount=Decimal("95000"), close_date=date(2026, 3, 25), probability=0.5, source_system="hubspot", source_id="301"),
]

sf_df = CRMContact.to_dataframe(sf_contacts)
hs_df = CRMContact.to_dataframe(hs_contacts)
accounts_df = CRMAccount.to_dataframe(accounts)
opps_df = CRMOpportunity.to_dataframe(opportunities)

print(f"Fixture contacts: Salesforce={len(sf_df)}, HubSpot={len(hs_df)}")
print(f"Fixture opportunities: {len(opps_df)}")


## 2. Cross-CRM deduplication

`crm_dedup_pipeline` creates deterministic canonical IDs from normalized names. In this fixture, Ada Lovelace appears in both systems and resolves to the same canonical ID.


In [ ]:
from siege_utilities.connectors import crm_dedup_pipeline

merge_table = crm_dedup_pipeline([sf_df, hs_df], name_columns=["first_name", "last_name"])
duplicates = merge_table[merge_table.duplicated("canonical_id", keep=False)]

print(f"Merge rows: {len(merge_table)}")
print(f"Canonical IDs: {merge_table['canonical_id'].nunique()}")
print("Cross-system duplicate names:", sorted(duplicates["normalized_name"].unique()))
display(merge_table.sort_values(["normalized_name", "source_system"]))


## 3. Geographic enrichment shape

The adapter stage prepares addresses for geocoding or boundary joins without performing geocoding itself. Postal codes remain strings so leading zeros would be preserved in other states.


In [ ]:
from siege_utilities.connectors import geographic_adapter

all_contacts = pd.concat([sf_df, hs_df], ignore_index=True, sort=False)
geo_ready = geographic_adapter(all_contacts)

print(f"Address rows ready for geocoding: {len(geo_ready)}")
print("Postal code dtype:", geo_ready["postal_code"].dtype)
display(geo_ready[["city", "state", "postal_code", "country"]])


## 4. Sales pipeline and tabular report shapes

Adapters reshape opportunity/account tables into stable DataFrames that chart/report builders can consume.


In [ ]:
from siege_utilities.connectors import pipeline_adapter, tabular_adapter

stage_order = ["Prospecting", "Proposal", "Negotiation", "Closed Won"]
pipeline = pipeline_adapter(opps_df, stage_column="stage", value_column="amount", stage_order=stage_order)
top_opps = tabular_adapter(
    opps_df,
    columns=["name", "stage", "amount", "close_date", "probability", "source_system"],
    rename={"name": "Opportunity", "amount": "Amount", "close_date": "Close Date"},
    sort_by="Amount",
    ascending=False,
)
top_accounts = tabular_adapter(
    accounts_df,
    columns=["name", "revenue", "employee_count", "source_system"],
    rename={"name": "Account", "revenue": "Revenue", "employee_count": "Employees"},
    sort_by="Revenue",
    ascending=False,
)

print("Pipeline stages:", pipeline["stage"].astype(str).tolist())
print("Pipeline total value:", int(pipeline["total_value"].sum()))
display(pipeline)
display(top_opps)
display(top_accounts)


## 5. Write-back planning, not execution

Governed notebooks must not mutate real systems. The final step creates an explicit write-back plan showing what would be sent to each source after operator review.


In [ ]:
sf_writeback = sf_df.merge(
    merge_table[["source_id", "canonical_id"]],
    on="source_id",
    how="left",
)[["source_system", "source_id", "canonical_id", "email"]]

hs_writeback = hs_df.merge(
    merge_table[["source_id", "canonical_id"]],
    on="source_id",
    how="left",
)[["source_system", "source_id", "canonical_id", "email"]]

writeback_plan = pd.concat([sf_writeback, hs_writeback], ignore_index=True).sort_values(["canonical_id", "source_system"])

print("Planned CRM updates only; no connector write methods are called.")
print(f"Rows requiring canonical_id write-back: {len(writeback_plan)}")
display(writeback_plan)


## Related

- Source: `siege_utilities/connectors/_models.py`, `siege_utilities/connectors/_dedup.py`, `siege_utilities/connectors/_adapters.py`, `siege_utilities/connectors/salesforce.py`, `siege_utilities/connectors/hubspot.py`
- Tests: `tests/test_connectors_errors.py`, `tests/test_connectors_salesforce_errors.py`, `tests/test_connectors_hubspot_errors.py`
- Notebook governance: `tests/test_notebook_hygiene.py`, `tests/test_notebooks.py`, `scripts/check_notebook_inventory.py`
